# MOF CO₂ 吸附量预测：以泛化 R² 为目标

从头建立可复现的实验：去重 → 固定独立验证集 → 相同五折 RMSE 调参 → OOF 融合 → 最后一次独立验证。

**本文件尚未使用竞赛数据运行，不包含预先编造的成绩。** 需要 `train.csv` 与 `test.csv`。不要将 Kaggle token 写入代码；使用本地数据或 Kaggle 官方认证。

OOF 分数用于选择参数、模型和融合权重，因此可能偏乐观；最后保留集才是主要的独立评估。该保留集只能在全部方案确定后查看一次。


In [ ]:
# 首次运行安装；完成后若内核提示，请重启内核。
%pip install pandas numpy scipy scikit-learn matplotlib xgboost catboost kagglehub

In [ ]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, ParameterSampler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

SEED = 42
THREADS = 2  # 不嵌套并行，避免搜索时内存耗尽
N_SPLITS = 5
N_CANDIDATES = 8  # 每种模型；首次可设为 2，再扩大搜索
SMOKE_TEST = False  # True 仅用于小规模流程检查，不能作为正式成绩
DATA_DIR = Path(os.environ.get("MOF_DATA_DIR", "data"))
DOWNLOAD_FROM_KAGGLE = False
TARGET = "CO2 uptake at 0.15 bar and 298K"
OUTPUT_DIR = Path("mof_r2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

import sklearn, xgboost, catboost
print({"sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
       "catboost": catboost.__version__})

## 1. 读取数据

默认读取当前目录的 `data/train.csv` 和 `data/test.csv`；也可修改 `DATA_DIR`。
若已具备 Kaggle 竞赛权限及官方认证，可将 `DOWNLOAD_FROM_KAGGLE=True`。下载被拒绝时，先在 Kaggle 接受竞赛规则，或手动下载 CSV；不要在 notebook 中填写密钥。


In [ ]:
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    DATA_DIR = Path(kagglehub.competition_download("che-1147-2026"))

for filename in ["train.csv", "test.csv"]:
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"请将 {filename} 放入 {DATA_DIR.resolve()}，或启用 Kaggle 下载。")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
assert TARGET in train.columns
assert train.columns.is_unique and test.columns.is_unique
assert "MOFname" in train.columns and "MOFname" in test.columns
print("原始数据:", train.shape, test.shape)

# 仅删除完全重复记录；不能按目标大小删除样本。
print("完全重复记录:", train.duplicated().sum())
train = train.drop_duplicates().reset_index(drop=True)
feature_cols = [c for c in test.columns if c not in {"MOFname", TARGET}]
assert feature_cols and all(c in train.columns for c in feature_cols)
assert not train[feature_cols].select_dtypes(exclude="number").columns.tolist(), "请先处理非数值特征"
assert not test[feature_cols].select_dtypes(exclude="number").columns.tolist()
X = train[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
y = pd.to_numeric(train[TARGET], errors="raise")
assert np.isfinite(y).all() and y.nunique() > 1
# 使用稳定内部列名，避免 XGBoost 对特殊字符的限制。
feature_map = dict(zip(feature_cols, [f"f{i:03d}" for i in range(len(feature_cols))]))
X = X.rename(columns=feature_map)
X_test = X_test.rename(columns=feature_map)
pd.Series(feature_map, name="internal_name").to_csv(OUTPUT_DIR / "feature_map.csv")
print("清理后:", X.shape, "缺失值:", X.isna().sum().sum())
print(y.describe())

## 2. 固定独立验证集

为避免重复结构泄漏，若去重后仍有相同 MOF 名称或相同特征向量，本流程会暂停，要求先确定分组策略，而不是悄悄随机拆分。若拥有金属/连接体/拓扑家族标签，还应另做按家族分组的验证；随机拆分衡量的是相同数据分布下的泛化。

所有候选模型使用相同开发集五折。固定保留集后，不依据它调参或改融合权重。


In [ ]:
assert not train["MOFname"].duplicated().any(), "同名 MOF 仍有不同记录，请用分组划分"
assert not X.duplicated().any(), "仍有相同特征的多条记录，请分组，先检查目标是否冲突"
idx_dev, idx_holdout = train_test_split(np.arange(len(X)), test_size=0.20, random_state=SEED)
X_dev = X.iloc[idx_dev].reset_index(drop=True)
y_dev = y.iloc[idx_dev].reset_index(drop=True)
X_holdout = X.iloc[idx_holdout].reset_index(drop=True)
y_holdout = y.iloc[idx_holdout].reset_index(drop=True)
folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X_dev))
np.savez(OUTPUT_DIR / "split_indices.npz", development=idx_dev, holdout=idx_holdout)
print("开发集 / 独立验证集:", len(idx_dev), len(idx_holdout))

def metrics(actual, pred):
    return {"RMSE": float(np.sqrt(mean_squared_error(actual, pred))),
            "MAE": float(mean_absolute_error(actual, pred)),
            "R2": float(r2_score(actual, pred))}

## 3. 模型与 early stopping

XGBoost 和 CatBoost 在**每个训练折内部**留出 15% 决定树数，再用该训练折全部样本重新拟合；外层评估折不参与 early stopping。最终拟合时取五折树数的中位数。

保持平方误差目标。在同一评估集上，最小化 RMSE 与最大化 R² 等价；这里搜索使用整体 OOF RMSE。ExtraTrees 的填补器只在相应训练折拟合。


In [ ]:
def make_model(kind, params, trees=None):
    if kind == "xgb":
        return XGBRegressor(**params, n_estimators=trees or (40 if SMOKE_TEST else 2200),
                            objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
                            random_state=SEED, n_jobs=THREADS)
    if kind == "cat":
        return CatBoostRegressor(**params, iterations=trees or (40 if SMOKE_TEST else 2200),
                                 loss_function="RMSE", random_seed=SEED, thread_count=THREADS,
                                 verbose=False, allow_writing_files=False, nan_mode="Min")
    if kind == "extra":
        return make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                             ExtraTreesRegressor(**params, n_estimators=40 if SMOKE_TEST else 500,
                                                 random_state=SEED, n_jobs=THREADS))
    raise ValueError(kind)

def fit_one_fold(kind, params, Xtr, ytr, seed):
    if kind == "extra":
        return make_model(kind, params).fit(Xtr, ytr), None
    it, ie = train_test_split(np.arange(len(Xtr)), test_size=0.15, random_state=seed)
    pilot = make_model(kind, params)
    if kind == "xgb":
        pilot.set_params(early_stopping_rounds=15 if SMOKE_TEST else 100)
        pilot.fit(Xtr.iloc[it], ytr.iloc[it], eval_set=[(Xtr.iloc[ie], ytr.iloc[ie])], verbose=False)
        trees = int(pilot.best_iteration) + 1
    else:
        pilot.fit(Xtr.iloc[it], ytr.iloc[it], eval_set=(Xtr.iloc[ie], ytr.iloc[ie]),
                  early_stopping_rounds=15 if SMOKE_TEST else 100, verbose=False)
        trees = int(pilot.tree_count_)
    model = make_model(kind, params, trees=trees)
    return model.fit(Xtr, ytr), trees

spaces = {
    "xgb": {"max_depth": [2, 3, 4, 5, 6], "learning_rate": [0.02, 0.04, 0.07],
            "min_child_weight": [3, 8, 15, 25], "subsample": [0.7, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0], "reg_alpha": [0, 0.1, 1.0],
            "reg_lambda": [3, 10, 30]},
    "cat": {"depth": [4, 5, 6, 7], "learning_rate": [0.02, 0.04, 0.07],
            "l2_leaf_reg": [3, 10, 30], "random_strength": [0.2, 1.0, 3.0]},
    "extra": {"max_features": [0.5, 0.8, 1.0], "min_samples_leaf": [1, 2, 4, 8],
              "max_depth": [None, 12, 20]}
}

In [ ]:
# CPU 运行可能需要较长时间；先减少 N_CANDIDATES 做正式的小搜索。
# SMOKE_TEST=True 仅检查程序能运行，结果不代表正式实验。
search_rows, best = [], {}
for kind, space in spaces.items():
    best_rmse = np.inf
    for candidate, params in enumerate(ParameterSampler(space, n_iter=1 if SMOKE_TEST else N_CANDIDATES,
                                                       random_state=SEED)):
        oof = np.full(len(X_dev), np.nan)
        tree_counts, fold_scores = [], []
        for fold, (tr, va) in enumerate(folds):
            model, trees = fit_one_fold(kind, params, X_dev.iloc[tr], y_dev.iloc[tr], SEED + fold)
            oof[va] = model.predict(X_dev.iloc[va])
            fold_scores.append(metrics(y_dev.iloc[va], oof[va]))
            if trees is not None:
                tree_counts.append(trees)
        assert np.isfinite(oof).all()
        score = metrics(y_dev, oof)
        search_rows.append({"model": kind, "candidate": candidate, **score,
                            "fold_RMSE_std": float(np.std([s["RMSE"] for s in fold_scores])),
                            "params": json.dumps(params)})
        if score["RMSE"] < best_rmse:
            best_rmse = score["RMSE"]
            best[kind] = {"params": params, "oof": oof.copy(), "trees": tree_counts,
                          "fold_scores": fold_scores, **score}
        print(kind, candidate + 1, score, flush=True)
        pd.DataFrame(search_rows).to_csv(OUTPUT_DIR / "search_results.csv", index=False)
comparison = pd.DataFrame([{ "model": k, **{m: v[m] for m in ["RMSE", "MAE", "R2"]}}
                          for k, v in best.items()]).sort_values("RMSE")
display(comparison)

## 4. 小范围 OOF 融合

比较单模型与少量预先定义的等权融合，避免用大量连续权重把 OOF 本身拟合过头。只有 OOF RMSE 优于最佳单模型至少 0.2% 才选择融合。该阈值是保守的选择规则，不是显著性检验。


In [ ]:
from itertools import combinations
names = list(best)
single = min(names, key=lambda k: best[k]["RMSE"])
chosen_names = (single,)
chosen_pred = best[single]["oof"].copy()
blend_rows = []
for size in range(2, len(names) + 1):
    for combo in combinations(names, size):
        pred = np.mean([best[k]["oof"] for k in combo], axis=0)
        score = metrics(y_dev, pred)
        blend_rows.append({"models": "+".join(combo), **score})
        if score["RMSE"] < metrics(y_dev, chosen_pred)["RMSE"] and score["RMSE"] < best[single]["RMSE"] * 0.998:
            chosen_names, chosen_pred = combo, pred
print("最终方案（查看保留集前确定）:", chosen_names)
display(pd.DataFrame(blend_rows).sort_values("RMSE"))
oof_table = pd.DataFrame({"Actual": y_dev, **{k: v["oof"] for k,v in best.items()},
                          "Selected": chosen_pred})
oof_table.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)
print("开发 OOF（存在模型选择偏差）:", metrics(y_dev, chosen_pred))

## 5. 诊断平方误差来源

按真实目标分组仅用于事后误差诊断，不能据此在预测时选择模型。高目标组存在偏差不自动证明缺失特征，也不能据此删除高值。


In [ ]:
diagnostics = pd.DataFrame({"Actual": y_dev, "Predicted": chosen_pred})
diagnostics["Residual"] = diagnostics.Actual - diagnostics.Predicted
diagnostics["SE"] = diagnostics.Residual ** 2
diagnostics["AE"] = diagnostics.Residual.abs()
diagnostics["Range"] = pd.cut(diagnostics.Actual, [-np.inf, 1, 2, 3, 4, 5, np.inf])
summary = diagnostics.groupby("Range", observed=True).agg(
    Count=("SE", "size"), SSE=("SE", "sum"), MSE=("SE", "mean"),
    MAE=("AE", "mean"), Bias=("Residual", "mean"))
summary["RMSE"] = np.sqrt(summary.MSE)
summary["SSE_share"] = summary.SSE / summary.SSE.sum() if summary.SSE.sum() else 0.0
display(summary)
summary.to_csv(OUTPUT_DIR / "error_by_range.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_dev, chosen_pred, s=10, alpha=0.4)
lo, hi = min(y_dev.min(), chosen_pred.min()), max(y_dev.max(), chosen_pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--")
axes[0].set(xlabel="Actual", ylabel="OOF prediction")
axes[1].scatter(chosen_pred, y_dev - chosen_pred, s=10, alpha=0.4)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="OOF prediction", ylabel="Residual")
plt.tight_layout()
plt.show()

## 6. 最后一次独立验证

只评估已选定的方案，不在这里挑选模型。若看完分数继续调参，该保留集就变成开发数据，需新的外部数据或嵌套 CV 才能做独立评估。此分数不能直接与旧 notebook 不同划分的 0.718 比较。


In [ ]:
def refit_selected(X_fit, y_fit, X_predict):
    predictions = []
    for kind in chosen_names:
        info = best[kind]
        trees = max(1, int(np.median(info["trees"]))) if info["trees"] else None
        model = make_model(kind, info["params"], trees=trees)
        model.fit(X_fit, y_fit)
        predictions.append(model.predict(X_predict))
    return np.mean(predictions, axis=0)

holdout_pred = refit_selected(X_dev, y_dev, X_holdout)
holdout_metrics = metrics(y_holdout, holdout_pred)
print("独立验证集:", holdout_metrics)
pd.DataFrame({"MOFname": train.iloc[idx_holdout]["MOFname"].to_numpy(),
              "Actual": y_holdout, "Predicted": holdout_pred}).to_csv(
                  OUTPUT_DIR / "holdout_predictions.csv", index=False)
report = {"smoke_test": SMOKE_TEST, "seed": SEED, "chosen_models": list(chosen_names),
          "holdout_metrics": holdout_metrics,
          "parameters": {k: {"params": v["params"], "fold_tree_counts": v["trees"]} for k,v in best.items()}}
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

## 7. 全量重训与提交文件

方案确定后使用全部已去重训练样本重训。沿用开发阶段选择的参数与树数，避免再用竞赛测试集调整。没有标签的 `test.csv` 无法计算 R²。

输出保留在 `mof_r2_outputs/`；提交前核对竞赛要求的列名及 `sample_submission.csv`。若为 smoke test，禁止生成正式提交。


In [ ]:
if SMOKE_TEST:
    print("Smoke test 完成；不生成竞赛提交。")
else:
    test_pred = refit_selected(X, y, X_test)
    assert len(test_pred) == len(test) and np.isfinite(test_pred).all()
    # 不根据保留集临时添加截断或其他后处理。
    sample_path = DATA_DIR / "sample_submission.csv"
    if sample_path.exists():
        submission = pd.read_csv(sample_path)
        assert len(submission) == len(test)
        assert TARGET in submission.columns and "MOFname" in submission.columns
        assert submission["MOFname"].astype(str).equals(test["MOFname"].astype(str)), "请先按 MOFname 对齐提交模板"
        submission[TARGET] = test_pred
    else:
        submission = pd.DataFrame({"MOFname": test["MOFname"], TARGET: test_pred})
        warnings.warn("没有 sample_submission.csv，请人工核对提交格式。")
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("已保存:", OUTPUT_DIR / "submission.csv")

## 后续实验顺序

1. 先完成上面的统一基线，检查各折波动和独立验证表现。
2. 在开发折内部比较全部 272 个特征与几何 + 金属中心 + 配体特征；不要在全数据上先筛选。
3. 核对物理单位后，再测试孔径比例、孔体积及金属描述符交互；每次加入一组。
4. 若要验证对新 MOF 家族的泛化，使用可靠结构家族标签另做 GroupKFold。

提高 R² 没有保证；应保留失败实验，不能通过换随机种子、删高误差样本或反复查看保留集来制造提升。
